<a href="https://colab.research.google.com/github/Akshara-codes/Movie-Recommendation-System/blob/main/Movie_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path1="/content/drive/MyDrive/Movie-recommendation-system/tmdb_5000_movies.csv"
path2="/content/drive/MyDrive/Movie-recommendation-system/tmdb_5000_credits.csv"

In [ ]:
movies = pd.read_csv(path1)
credits = pd.read_csv(path2)
movies=movies.merge(credits,on='title')

In [ ]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.isnull().sum()

,0
movie_id,0
title,0
overview,3
genres,0
keywords,0
cast,0
crew,0


In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

np.int64(0)

In [ ]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [ ]:
# [{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]
# we need to convert it in this format : ['Action','Adventure','Fantasy','Science fiction']

In [ ]:
import ast  # To convert string to int
# Helper Function
def convert(obj):
  L=[]
  for i in ast.literal_eval(obj):
   L.append(i['name'])
  return L
movies['genres']=movies['genres'].apply(convert)

In [ ]:
movies['keywords']=movies['keywords'].apply(convert)

In [ ]:
# to convert cast into top 3
def convert3(obj):
  L=[]
  counter=0
  for i in ast.literal_eval(obj):
    if counter!=3:
      L.append(i['name'])
      counter+=1
    else:
      break
  return L
movies['cast']=movies['cast'].apply(convert3)

In [ ]:
# to convert crew to extract director
def fetch_director(obj):
  L=[]
  for i in ast.literal_eval(obj):
    if i['job']=='Director':
      L.append(i['name'])
      break
  return L
movies['crew']=movies['crew'].apply(fetch_director)


In [ ]:
# overview is a str so converting it into list to conactinate with others
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [ ]:
# to transform the columns to remove spaces to remove ambiguity caused due to same names or words
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies['tags']=movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
new_df=movies[['movie_id','title','tags']]

In [ ]:
# now convert the tags lists into strings
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

/tmp/ipykernel_11314/901568765.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))


In [ ]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())


/tmp/ipykernel_11314/3329932311.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:x.lower())


In [ ]:
# Stemming - to perform similar words
import nltk
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [ ]:
def stem(text):
  y=[]
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [ ]:
new_df['tags']=new_df['tags'].apply(stem)

/tmp/ipykernel_11314/3514595201.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(stem)


In [ ]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


In [ ]:
# Tags Vectorization
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vectors=cv.fit_transform(new_df['tags']).toarray()

In [ ]:
cv.get_feature_names_out()        # to see the words formed

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vectors)

In [ ]:
similarity[1]

array([0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
       0.02615329])

In [ ]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x: x[1])[1:6]

[(1214, np.float64(0.28676966733820225)),
 (2405, np.float64(0.26901379342448517)),
 (3728, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574))]

In [ ]:
new_df[new_df['title']=="Avatar"].index[0]

np.int64(0)

In [ ]:
def recommend(movie):
  movie_index = new_df[new_df['title'] == movie].index[0]
  distances = similarity[movie_index]
  # to hold the index
  movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
  for i in movies_list:
    print(new_df.iloc[i[0]].title)


In [ ]:
recommend('Batman Begins')

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


In [ ]:
import pickle
pickle.dump(new_df,open('movies.pkl','wb'))

In [ ]:
import numpy as np

def compress_similarity(sim_matrix, top_n=20):
    compressed = []
    for row in sim_matrix:
        top_indices = np.argsort(row)[::-1][:top_n+1]  # top N+1 (includes self)
        top_scores = row[top_indices]
        compressed.append((top_indices, top_scores))
    return compressed

compressed_sim = compress_similarity(similarity)
pickle.dump(compressed_sim, open('similarity_compressed.pkl', 'wb'))
# This will be only ~5-10MB!

In [ ]:
import pickle
movies = pickle.load(open('movies.pkl', 'rb'))
print(movies.columns.tolist())
print(movies.head(2))

['movie_id', 'title', 'tags']
   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   

                                                tags  
0  in the 22nd century, a parapleg marin is dispa...  
1  captain barbossa, long believ to be dead, ha c...  


In [ ]:
import requests
import pickle
import time

API_KEY = "0cff35dc37984f113d33747ae2d9fbc1"  # 🔑 replace this

movies = pickle.load(open('movies.pkl', 'rb'))

def fetch_poster(movie_id):
    try:
        url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={API_KEY}"
        response = requests.get(url, timeout=5)
        data = response.json()
        poster_path = data.get('poster_path', '')
        if poster_path:
            return f"https://image.tmdb.org/t/p/w500{poster_path}"
        return None
    except:
        return None  # if any movie fails, just skip it

# Fetch all posters (will take 2-3 minutes)
posters = []
for i, movie_id in enumerate(movies['movie_id']):
    poster = fetch_poster(movie_id)
    posters.append(poster)
    if i % 100 == 0:
        print(f"Progress: {i}/{len(movies)} done...")
    time.sleep(0.1)  # avoid hitting API rate limit

movies['poster_url'] = posters

# Save updated file
pickle.dump(movies, open('movies_with_posters.pkl', 'wb'))
print("✅ Done! Total movies:", len(movies))
print("✅ Posters found:", movies['poster_url'].notna().sum())

Progress: 0/4806 done...
Progress: 100/4806 done...
Progress: 200/4806 done...
Progress: 300/4806 done...
Progress: 400/4806 done...
Progress: 500/4806 done...
Progress: 600/4806 done...
Progress: 700/4806 done...
Progress: 800/4806 done...
Progress: 900/4806 done...
Progress: 1000/4806 done...
Progress: 1100/4806 done...
Progress: 1200/4806 done...
Progress: 1300/4806 done...
Progress: 1400/4806 done...
Progress: 1500/4806 done...
Progress: 1600/4806 done...
Progress: 1700/4806 done...
Progress: 1800/4806 done...
Progress: 1900/4806 done...
Progress: 2000/4806 done...
Progress: 2100/4806 done...
Progress: 2200/4806 done...
Progress: 2300/4806 done...
Progress: 2400/4806 done...
Progress: 2500/4806 done...
Progress: 2600/4806 done...
Progress: 2700/4806 done...
Progress: 2800/4806 done...
Progress: 2900/4806 done...
Progress: 3000/4806 done...
Progress: 3100/4806 done...
Progress: 3200/4806 done...
Progress: 3300/4806 done...
Progress: 3400/4806 done...
Progress: 3500/4806 done...
Prog